In [ ]:
# ============================================================
# LANGGRAPH MULTI-AGENT SYSTEM
# Google Colab - One Cell
# ============================================================

# 1️⃣ INSTALL REQUIRED LIBRARIES
!pip -q install langgraph langchain-groq

# ============================================================
# 2️⃣ IMPORT LIBRARIES
# ============================================================

import os
from typing import TypedDict

from getpass import getpass
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END


# ============================================================
# 3️⃣ ENTER YOUR GROQ API KEY
# ============================================================

print("Enter your Groq API Key:")
GROQ_API_KEY = getpass("Groq API Key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


# ============================================================
# 4️⃣ CREATE THE AI MODEL
# ============================================================

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)


# ============================================================
# 5️⃣ CREATE SHARED STATE
# ============================================================

class State(TypedDict):
    topic: str
    research: str
    article: str
    review: str


# ============================================================
# 6️⃣ RESEARCH AGENT
# ============================================================

def research_agent(state):

    topic = state["topic"]

    prompt = f"""
You are a Research Agent.

Research the topic:

{topic}

Provide:
1. Simple definition
2. Important concepts
3. Real-world applications
4. Important facts

Explain everything for a beginner.
"""

    response = llm.invoke(prompt)

    print("\n🔎 RESEARCH AGENT FINISHED")

    return {
        "research": response.content
    }


# ============================================================
# 7️⃣ WRITER AGENT
# ============================================================

def writer_agent(state):

    research = state["research"]

    prompt = f"""
You are a Writing Agent.

Use the research below to create
a beginner-friendly explanation.

RESEARCH:
{research}

Requirements:
- Use simple English
- Use headings
- Give examples
- Keep the explanation easy to understand
"""

    response = llm.invoke(prompt)

    print("✍️ WRITER AGENT FINISHED")

    return {
        "article": response.content
    }


# ============================================================
# 8️⃣ REVIEWER AGENT
# ============================================================

def reviewer_agent(state):

    article = state["article"]

    prompt = f"""
You are a Review Agent.

Review the following article:

{article}

Check:
1. Is the explanation correct?
2. Is it beginner friendly?
3. Are important points missing?
4. Is anything confusing?

Give a short review and suggestions.
"""

    response = llm.invoke(prompt)

    print("🔍 REVIEWER AGENT FINISHED")

    return {
        "review": response.content
    }


# ============================================================
# 9️⃣ CREATE LANGGRAPH
# ============================================================

graph = StateGraph(State)


# Add agents as nodes

graph.add_node(
    "researcher",
    research_agent
)

graph.add_node(
    "writer",
    writer_agent
)

graph.add_node(
    "reviewer",
    reviewer_agent
)


# ============================================================
# 🔟 CONNECT THE AGENTS
# ============================================================

graph.add_edge(
    START,
    "researcher"
)

graph.add_edge(
    "researcher",
    "writer"
)

graph.add_edge(
    "writer",
    "reviewer"
)

graph.add_edge(
    "reviewer",
    END
)


# ============================================================
# 1️⃣1️⃣ COMPILE THE GRAPH
# ============================================================

app = graph.compile()


# ============================================================
# 1️⃣2️⃣ ASK USER FOR A TOPIC
# ============================================================

topic = input(
    "\nEnter a topic for the AI agents to research: "
)


# ============================================================
# 1️⃣3️⃣ RUN THE MULTI-AGENT SYSTEM
# ============================================================

result = app.invoke({

    "topic": topic,

    "research": "",

    "article": "",

    "review": ""
})


# ============================================================
# 1️⃣4️⃣ DISPLAY FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("🔎 RESEARCH AGENT OUTPUT")
print("=" * 70)

print(result["research"])


print("\n")
print("=" * 70)
print("✍️ WRITER AGENT OUTPUT")
print("=" * 70)

print(result["article"])


print("\n")
print("=" * 70)
print("🔍 REVIEWER AGENT OUTPUT")
print("=" * 70)

print(result["review"])


print("\n")
print("=" * 70)
print("✅ MULTI-AGENT WORKFLOW COMPLETED")
print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.6 MB/s eta 0:00:00
Enter your Groq API Key:
Groq API Key: ··········

Enter a topic for the AI agents to research: Stars

🔎 RESEARCH AGENT FINISHED
✍️ WRITER AGENT FINISHED
🔍 REVIEWER AGENT FINISHED


🔎 RESEARCH AGENT OUTPUT
## ⭐️ Stars – A Beginner‑Friendly Research Overview  

Below is a concise, easy‑to‑read guide that covers what stars are, the key ideas you need to know, how they matter in everyday life, and some “wow‑factor” facts that make them fascinating.

---

## 1. Simple Definition  

**Star (noun):**  
A massive, glowing ball of hot gas (mostly hydrogen and helium) held together by its own gravity. Inside a star, nuclear fusion constantly converts hydrogen into helium, releasing huge amounts of energy as light and heat. The Sun is the star closest to Earth.

---

## 2. Important Concepts  

| Concept | What It Means (in plain language) | Why It Matters |
|---------|-----------------------------------|-----------